# Practical 8 – MongoDB Basics

**Student:** JAISWAL VICKY VIRJU &nbsp;|&nbsp; **Roll No:** TCS2526021 &nbsp;|&nbsp; **Semester:** V

## Concept
**MongoDB** is a NoSQL document-oriented database that stores data as **JSON-like BSON documents** inside **Collections** (analogous to SQL tables).
Key advantages: flexible schema, horizontal scalability, and native JSON support.
- `mongod` → starts the MongoDB **server** (manages database files, listens for requests)
- `mongo` → opens the MongoDB **shell** (client interface to run CRUD commands)
- **MapReduce** → a data processing technique that maps each document to a key-value pair, then reduces all values for the same key (e.g., summing sales per product).

## Shell Commands Reference
| Command | Purpose |
|---|---|
| `use <dbname>` | Switch to (or create) a database |
| `db.Collection.insertMany([…])` | Insert multiple documents |
| `db.Collection.find({})` | Retrieve all documents |
| `db.Collection.find({}, {field:1, _id:0})` | Project specific fields |
| `db.Collection.find().sort({field:-1})` | Sort descending |
| `db.Collection.find({$or:[…]})` | OR filter condition |
| `db.Collection.update({filter},{$set:{…}},{multi:true})` | Update many |
| `db.Collection.remove({filter})` | Delete matching documents |
| `db.Collection.mapReduce(map, reduce, {out:…})` | Run MapReduce job |

## Code Flow
```
Setup:
1. Install MongoDB → start mongod (server)
2. Open mongo shell (client)

Task 1 – Staff Collection:
3. use Institution
4. insertMany([10 employee docs])
5. find({})                          ← all docs
6. find({},{empid:1,designation:1})  ← projection
7. find().sort({salary:-1})          ← descending sort
8. find({$or:[{designation:'Manager'},{salary:{$gt:50000}}]})
9. update({designation:'Accountant'},{$set:{salary:45000}},{multi:true})
10. remove({salary:{$gt:100000}})

Task 2 – Student Collection: (same CRUD pattern)

Task 3 – MapReduce on Sales:
11. insertMany([6 sales docs])
12. Define mapFunction  → emit(product, amount)
13. Define reduceFunction → Array.sum(values)
14. mapReduce(map, reduce, {out:'TotalSales'})
15. db.TotalSales.find()
```

## MongoDB Shell Commands (copy & run in `mongo` shell)

In [ ]:
# ── NOTE ──────────────────────────────────────────────────────────────────
# MongoDB shell commands cannot run inside a Python notebook directly.
# This cell shows the equivalent Python simulation using a dict-based store,
# so you can understand the logic. Run the actual commands in the mongo shell.
# ──────────────────────────────────────────────────────────────────────────

# Python simulation of MongoDB-like operations

Staff = [
    {"empid":1,  "empname":"Janaki",   "salary":90000,  "designation":"Manager"},
    {"empid":2,  "empname":"Shree",    "salary":55000,  "designation":"HR"},
    {"empid":3,  "empname":"Bhaskar",  "salary":45000,  "designation":"Accountant"},
    {"empid":4,  "empname":"Niharika", "salary":70000,  "designation":"Manager"},
    {"empid":5,  "empname":"Rahul",    "salary":35000,  "designation":"Clerk"},
    {"empid":6,  "empname":"Piyush",   "salary":48000,  "designation":"Accountant"},
    {"empid":7,  "empname":"Vikas",    "salary":60000,  "designation":"Manager"},
    {"empid":8,  "empname":"Sneha",    "salary":32000,  "designation":"Clerk"},
    {"empid":9,  "empname":"Arjun",    "salary":110000, "designation":"Director"},
    {"empid":10, "empname":"Manaki",   "salary":85000,  "designation":"Manager"},
]

print("All documents inserted.")
print(f"Total documents: {len(Staff)}")

In [ ]:
import pandas as pd

# Display all documents
df_staff = pd.DataFrame(Staff)
print("=== All Documents ===")
print(df_staff.to_string(index=False))

In [ ]:
# Project only empid and designation
print("=== empid & designation only ===")
print(df_staff[['empid','designation']].to_string(index=False))

In [ ]:
# Sort descending by salary
print("=== Sorted by Salary (Descending) ===")
print(df_staff.sort_values('salary', ascending=False).to_string(index=False))

In [ ]:
# OR condition: Manager OR salary > 50000
mask = (df_staff['designation'] == 'Manager') | (df_staff['salary'] > 50000)
print("=== Manager OR Salary > 50,000 ===")
print(df_staff[mask].to_string(index=False))

In [ ]:
# Update Accountant salary to 45000
df_staff.loc[df_staff['designation'] == 'Accountant', 'salary'] = 45000
print("=== After updating Accountant salary to ₹45,000 ===")
print(df_staff[df_staff['designation'] == 'Accountant'].to_string(index=False))

In [ ]:
# Remove employees with salary > 100000
before = len(df_staff)
df_staff = df_staff[df_staff['salary'] <= 100000].reset_index(drop=True)
print(f"Removed {before - len(df_staff)} document(s) with salary > ₹1,00,000")
print(df_staff.to_string(index=False))

## Task 3 – MapReduce: Total Sales by Product

In [ ]:
# Sales data
sales = [
    {"_id":1, "product":"apple",   "amount":100},
    {"_id":2, "product":"banana",  "amount":150},
    {"_id":3, "product":"apple",   "amount":200},
    {"_id":4, "product":"oranges", "amount":100},
    {"_id":5, "product":"banana",  "amount":350},
    {"_id":6, "product":"oranges", "amount":200},
]

# MAP: emit (product, amount) for each document
mapped = [(doc['product'], doc['amount']) for doc in sales]
print("Mapped pairs:", mapped)

In [ ]:
from collections import defaultdict

# REDUCE: sum amounts per product
reduced = defaultdict(int)
for key, value in mapped:
    reduced[key] += value

result = pd.DataFrame(list(reduced.items()), columns=['product','total_sales'])
result = result.sort_values('total_sales', ascending=False).reset_index(drop=True)
print("=== MapReduce Result: Total Sales by Product ===")
print(result.to_string(index=False))

## Actual MongoDB Shell Commands
```js
// Task 1 – Staff
use Institution
db.Staff.insertMany([...])
db.Staff.find({})
db.Staff.find({},{empid:1,designation:1,_id:0})
db.Staff.find().sort({salary:-1})
db.Staff.find({$or:[{designation:'Manager'},{salary:{$gt:50000}}]})
db.Staff.update({designation:'Accountant'},{$set:{salary:45000}},{multi:true})
db.Staff.remove({salary:{$gt:100000}})

// Task 2 – Student (same CRUD pattern)
use Institution
db.Student.insertMany([...])
db.Student.find({})
db.Student.find().sort({TotalMarks:-1})
db.Student.find({$or:[{Class:'MSc'},{TotalMarks:{$gt:400}}]})
db.Student.remove({TotalMarks:{$lt:200}})

// Task 3 – MapReduce
db.sales.insertMany([...])
var mapFunction    = function(){ emit(this.product, this.amount); }
var reduceFunction = function(key, values){ return Array.sum(values); }
db.sales.mapReduce(mapFunction, reduceFunction, {out:'TotalSales'})
db.TotalSales.find()
```